# ED Agent Mesh — Patched Synthetic Runner (Day 6)
This notebook includes:
- Stage-1 guardrails for Pandas and LLM proposals
- Policy Layer (cooldowns, idempotency, edge-triggered capacity, repeat policy)
- A minimal synthetic end-to-end scenario demonstrating correct behavior

Run cells from top to bottom.

In [ ]:

# --- Stage 1 Safety Guardrails (compact) ---
import pandas as pd

pd.options.mode.chained_assignment = 'raise'  # loud on chained indexing

def load_ed_data(path_or_buf):
    df = pd.read_csv(path_or_buf).convert_dtypes()
    if 'ts' in df.columns:
        df['ts'] = pd.to_datetime(df['ts'], utc=True, errors='raise')
    if 'encounter_id' in df.columns:
        assert df['encounter_id'].is_unique, "Duplicate encounter_id detected!"
    return df

def validate_df(df, name="DataFrame"):
    if 'encounter_id' in df.columns:
        assert df['encounter_id'].is_unique, f"{name}: duplicate encounter_id"
    if 'ts' in df.columns:
        assert df['ts'].is_monotonic_increasing, f"{name}: timestamps not monotonic"
    print(f"[OK] {name} passed Stage 1 checks.")

# LLM proposal minimal validator (allowed actions + required fields)
import json, hashlib
from typing import Dict, Any, List

ALLOWED_ACTIONS = {"page_team", "order_ct", "request_labs", "hold_bed", "open_case", "order_ecg", "ed_hold", "icu_downgrade", "bed.request"}

PARAM_SCHEMAS: Dict[str, Dict[str, str]] = {
    "page_team":    {"team": "str", "priority": "enum:STAT|URGENT", "reason": "str"},
    "order_ct":     {"protocol": "str", "priority": "enum:STAT|ROUTINE"},
    "request_labs": {"panel_id": "str", "priority": "enum:STAT|ROUTINE"},
    "hold_bed":     {"service": "enum:ICU|CARDS|NEPHRO|OBGYN|MED", "level": "enum:WARD|STEPDOWN|ICU"},
    "open_case":    {"patient_ref": "str"},
    "order_ecg":    {"priority": "enum:STAT|ROUTINE"},
    "ed_hold":      {"reason": "str"},
    "icu_downgrade":{"to_level": "enum:WARD|STEPDOWN"}
}

MUST_REQUIRE_PERMIT = {"order_ct", "request_labs", "hold_bed", "icu_downgrade"}

def _ensure_type(name: str, val: Any, want: str):
    if want == "str" and not isinstance(val, str):
        raise ValueError(f"param.{name} must be str")
    if want == "int" and not isinstance(val, int):
        raise ValueError(f"param.{name} must be int")
    if want == "float" and not isinstance(val, (int, float)):
        raise ValueError(f"param.{name} must be float")

def _ensure_enum(name: str, val: Any, options: List[str]):
    if not isinstance(val, str) or val not in options:
        raise ValueError(f"param.{name} must be one of {options}")

def _validate_params(action: str, params: Dict[str, Any]) -> Dict[str, Any]:
    if action not in PARAM_SCHEMAS:
        if params:
            raise ValueError(f"no schema for action '{action}', params must be empty for now")
        return {}
    schema = PARAM_SCHEMAS[action]
    extra = set(params.keys()) - set(schema.keys())
    if extra:
        raise ValueError(f"unexpected params for {action}: {sorted(extra)}")
    missing = [k for k in schema.keys() if k not in params]
    if missing:
        raise ValueError(f"missing params for {action}: {missing}")
    for k, rule in schema.items():
        if rule.startswith("enum:"):
            options = rule.split(":",1)[1].split("|")
            _ensure_enum(k, params[k], options)
        else:
            _ensure_type(k, params[k], rule)
    return params

def _hash_params(encounter_id: str, action: str, params: Dict[str, Any]) -> str:
    blob = json.dumps({"encounter_id": encounter_id, "action": action, "params": params}, sort_keys=True)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()

def parse_and_validate_proposal(raw: str, encounter_id: str) -> Dict[str, Any]:
    d = json.loads(raw)
    for key in ("action", "params", "requires_permit", "justification"):
        if key not in d:
            raise ValueError(f"missing required field: {key}")
    if d["action"] not in ALLOWED_ACTIONS:
        raise ValueError(f"action '{d['action']}' not allowed")
    d["params"] = _validate_params(d["action"], d["params"])
    if d["action"] in MUST_REQUIRE_PERMIT and d["requires_permit"] is not True:
        raise ValueError(f"action '{d['action']}' must require permit")
    return d

print("[Stage 1 loaded: pandas guardrails + minimal LLM validator]")


In [ ]:

# ==== Policy Layer Patch (cooldowns, idempotency, edge-trigger, repeats) ====
from typing import Callable, Dict, Any, Tuple
from collections import defaultdict
import time, json, hashlib

def log_event(kind: str, action: str = "", note: str = "") -> None:
    print(f"{kind:<6} | {action} | {note}")

CANON = {
    "proposal.order.ecg":       "order_ecg",
    "proposal.order.labs":      "order_labs",
    "proposal.order.ct":        "order_ct",
    "proposal.bed.request":     "bed.request",
    "proposal.ed_hold":         "ed_hold",
    "proposal.icu_downgrade":   "icu_downgrade",
}

COOLDOWN_SEC = {
    "bed.request":  30 * 60,
    "ed_hold":      10 * 60,
    "order_ct":     365 * 24 * 3600,
    "order_ecg":    2 * 3600,
    "order_labs":   2 * 3600,
    "icu_downgrade": 60 * 60,
}

REPEAT_POLICY = {
    "order_ct":   {"allow": False},
    "order_ecg":  {"allow": True, "interval_sec": 2 * 3600},
    "order_labs": {"allow": True, "interval_sec": 2 * 3600,
                   "by_panel": {"trop_panel": 3 * 3600}}
}

_last_fired     = defaultdict(float)
_completed      = set()
_last_done      = {}
awaiting_capacity = defaultdict(bool)

def _idem_key(enc: str, canon: str, params: Dict[str, Any]) -> str:
    blob = json.dumps({"e": enc, "a": canon, "p": params}, sort_keys=True)
    return hashlib.sha256(blob.encode()).hexdigest()

def _params_sig(params: Dict[str, Any]) -> str:
    return hashlib.sha1(json.dumps(params, sort_keys=True).encode()).hexdigest()

def on_capacity_change(encounter_id: str, icu_free_beds: int) -> None:
    awaiting_capacity[encounter_id] = (icu_free_beds == 0)

def _can_emit(enc: str, canon: str, params: Dict[str, Any], now=None) -> Tuple[bool, str, str]:
    now = now or time.time()
    key = _idem_key(enc, canon, params)
    if key in _completed:
        return False, "already completed", key
    cd = COOLDOWN_SEC.get(canon, 0)
    if now - _last_fired[key] < cd:
        return False, f"cooldown {int(cd - (now - _last_fired[key]))}s", key
    if canon == "bed.request" and awaiting_capacity.get(enc, False):
        return False, "awaiting capacity change", key
    return True, "ok", key

def _can_repeat(enc: str, canon: str, params: Dict[str, Any], ctx: Dict[str, Any], now=None) -> Tuple[bool, str]:
    now = now or time.time()
    rule = REPEAT_POLICY.get(canon, {"allow": False})
    sig  = _params_sig(params)
    rk   = (enc, canon, sig)
    if canon == "order_ct" and ctx.get("new_indication", False):
        return True, "new indication"
    if not rule.get("allow", False):
        return (rk not in _last_done), "repeat not allowed"
    interval = rule.get("interval_sec")
    if "by_panel" in rule:
        interval = rule["by_panel"].get(params.get("panel_id"), interval)
    last = _last_done.get(rk, 0)
    if (now - last) < (interval or 0):
        return False, f"repeat cooldown {int((interval or 0) - (now - last))}s"
    return True, "ok"

def policy_emit(encounter_id: str, action: str, params: Dict[str, Any],
                ctx: Dict[str, Any], do_emit: Callable[[str, Dict[str, Any]], None]) -> bool:
    canon = CANON.get(action, action)
    ok1, reason = _can_repeat(encounter_id, canon, params, ctx)
    ok2, why2, key = _can_emit(encounter_id, canon, params)
    if not ok1:
        log_event("block", action, reason); return False
    if not ok2:
        log_event("block", action, why2);   return False
    do_emit(action, params)
    _last_fired[key] = time.time()
    log_event("action", action, "emitted")
    return True

def policy_complete(encounter_id: str, action: str, params: Dict[str, Any]) -> None:
    canon = CANON.get(action, action)
    key   = _idem_key(encounter_id, canon, params)
    _completed.add(key)
    _last_done[(encounter_id, canon, _params_sig(params))] = time.time()
    log_event("audit", action, "completed")

print("[Policy Layer ready: cooldowns, idempotency, edge-trigger, repeats]")


In [ ]:

# --- Minimal Event Bus + raw_emit + Synthetic Runner ---
from collections import deque

class EventBus:
    def __init__(self):
        self.queue = deque()
        self.log = []
    def propose(self, action, params):
        self.queue.append((action, params))
        self.log.append(("propose", action, params))
        log_event("emit", action, str(params))

eventbus = EventBus()

def raw_emit(action, params):
    eventbus.propose(action, params)

class Capacity:
    def __init__(self, icu_total=2, icu_occupied=2):
        self.icu_total = icu_total
        self.icu_occupied = icu_occupied
    @property
    def icu_free(self):
        return max(0, self.icu_total - self.icu_occupied)

capacity = Capacity(icu_total=2, icu_occupied=2)

def run_polytrauma(encounter_id="ED-TEST-001"):
    print(f"Mesh online. ICU capacity: total={capacity.icu_total}, occupied={capacity.icu_occupied}, free={capacity.icu_free}")
    ctx = {"icu_free_beds": capacity.icu_free}
    # ECG
    if policy_emit(encounter_id, "proposal.order.ecg", {"priority":"STAT"}, ctx, raw_emit):
        policy_complete(encounter_id, "proposal.order.ecg", {"priority":"STAT"})
    # Labs
    if policy_emit(encounter_id, "proposal.order.labs", {"panel_id":"trop_panel","priority":"STAT"}, ctx, raw_emit):
        policy_complete(encounter_id, "proposal.order.labs", {"panel_id":"trop_panel","priority":"STAT"})
    # CT
    if policy_emit(encounter_id, "proposal.order.ct", {"protocol":"head_trauma","priority":"STAT"}, ctx, raw_emit):
        policy_complete(encounter_id, "proposal.order.ct", {"protocol":"head_trauma","priority":"STAT"})
    # Try bed request (should block while full)
    policy_emit(encounter_id, "proposal.bed.request", {"service":"ICU"}, ctx, raw_emit)
    # ED hold as fallback
    if policy_emit(encounter_id, "proposal.ed_hold", {"reason":"ICU full / flow safety"}, ctx, raw_emit):
        policy_complete(encounter_id, "proposal.ed_hold", {"reason":"ICU full / flow safety"})
    # Free one ICU bed
    capacity.icu_occupied -= 1
    on_capacity_change(encounter_id, capacity.icu_free)
    ctx = {"icu_free_beds": capacity.icu_free}
    # Now bed request should emit once
    if policy_emit(encounter_id, "proposal.bed.request", {"service":"ICU"}, ctx, raw_emit):
        policy_complete(encounter_id, "proposal.bed.request", {"service":"ICU"})
    # Attempt to re-order CT (should be blocked)
    policy_emit(encounter_id, "proposal.order.ct", {"protocol":"head_trauma","priority":"STAT"}, ctx, raw_emit)
    print("— run complete —")

run_polytrauma()
